## Exercise 2.1

In [3]:
import tiktoken

text = "Akwirw ier"
bpe_tokenizer = tiktoken.get_encoding("gpt2")

token_ids = bpe_tokenizer.encode(text)
print(token_ids)

mappings = {token_id: bpe_tokenizer.decode([token_id]) for token_id in token_ids}
print(mappings)

decoded_tokens = bpe_tokenizer.decode(token_ids)
print(decoded_tokens)

[33901, 86, 343, 86, 220, 959]
{33901: 'Ak', 86: 'w', 343: 'ir', 220: ' ', 959: 'ier'}
Akwirw ier


In [4]:
import torch

from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = torch.tensor(token_ids[i:i+max_length])
            target_chunk = torch.tensor(token_ids[i+1:i+max_length+1])

            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    

def create_dataloader(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [7]:
verdict_path = "the-verdict.txt"

with open(verdict_path, "r", encoding="utf-8") as f:
    text = f.read()

dataloader = create_dataloader(text, batch_size=1, max_length=4, stride=1, shuffle=False)
next(iter(dataloader))

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

## Exercise 2.2

In [14]:
dataloader = create_dataloader(text, batch_size=1, max_length=2, stride=2, shuffle=False)
print(next(iter(dataloader)))

dataloader = create_dataloader(text, batch_size=1, max_length=8, stride=2, shuffle=False)
next(iter(dataloader))

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]


[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]),
 tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]

In [16]:
from torch.nn import Embedding

vocab_size = 50257
output_size = 256

embedding_layer = Embedding(vocab_size, output_size)

max_length = 4

dataloader = create_dataloader(text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)


In [17]:
context_length = max_length
pos_embedding_layer = Embedding(context_length, output_size)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
pos_embeddings

tensor([[-0.9742,  0.7024,  1.6606,  ...,  1.8827, -0.5992,  1.7251],
        [ 0.7939, -0.6538, -0.9689,  ...,  1.3654,  0.0691, -0.2018],
        [-0.3763, -0.5589,  0.5567,  ..., -0.1774,  0.0666, -1.9151],
        [-0.3475,  0.5572, -1.0025,  ...,  1.2004,  1.4369, -0.2788]],
       grad_fn=<EmbeddingBackward0>)